# 🎮 MO-PPO Super Mario Bros - Kaggle Runner
### 🏆 Group 1 - Đồ án Cuối kỳ Học Tăng Cường (Reinforcement Learning)

Notebook này được tối ưu sẵn 100% để chạy trên **Kaggle Notebooks (GPU NVIDIA P100 hoặc T4)**.
Tích hợp sẵn tự động clone code, cài đặt thư viện khóa phiên bản, đăng nhập Weights & Biases (WandB), huấn luyện, đánh giá và xem video trực tiếp.

## 1. Kiểm tra thông tin GPU và môi trường Kaggle
> Đảm bảo **Accelerator: GPU T4** (hoặc P100) và **Internet: ON** trong bảng Settings bên phải.

In [ ]:
!nvidia-smi
!pwd

## 2. Tải mã nguồn mới nhất từ GitHub
Tự động làm sạch thư mục cũ và clone bản cập nhật mới nhất đầy đủ tất cả các phân hệ.

In [ ]:
%cd /kaggle/working
!rm -rf DRL-Mo-Supermario-group-1-test
!git clone https://github.com/TTins1702/DRL-Mo-Supermario-group-1-test.git
%cd /kaggle/working/DRL-Mo-Supermario-group-1-test
!ls -la

## 3. Cài đặt các gói phụ thuộc (Khóa numpy < 2.0)
> **LƯU Ý QUAN TRỌNG:** Bắt buộc cài `numpy==1.26.4` để tương thích hoàn toàn với `nes-py` và `gym-super-mario-bros`.

In [ ]:
!pip install "numpy==1.26.4"
!pip install nes-py==8.2.1 gym-super-mario-bros==7.4.0 "gymnasium>=0.29.0,<1.0.0" mo-gymnasium>=1.0.0
!pip install rich tqdm pyyaml tensorboard opencv-python imageio imageio-ffmpeg wandb

## 4. Chạy kiểm thử tự động toàn bộ 4 phân hệ (PyTest Suite)
Kiểm tra tính đúng đắn của Môi trường (Env), Mô hình CNN (Model), Thuật toán MO-PPO (Agent) và Hạ tầng (Utils).

In [ ]:
!pytest -v tests/

## 5. Kết nối Weights & Biases (WandB) để theo dõi thời gian thực & xem Video
Tự động đọc `WANDB_API_KEY` từ Kaggle Secrets (Add-ons -> Secrets).

In [ ]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("✅ Đã kết nối Weights & Biases thành công!")
except Exception as e:
    print(f"⚠️ Không đọc được WANDB_API_KEY từ Secrets: {e}")
    print("Vui lòng chạy wandb.login() thủ công hoặc tick chọn WANDB_API_KEY ở cột Secrets bên phải.")
    wandb.login()

## 6. Bắt đầu Huấn luyện Baseline 8 Điểm (MO-PPO Super Mario)
Sử dụng vector trọng số w = [0.65, 0.10, 0.20, 0.02, 0.03] ưu tiên qua màn 1-1.
Sau mỗi 25 lần cập nhật, video Mario chơi sẽ được tự động upload trực tiếp lên WandB!

In [ ]:
!python train.py \
  --config configs/base_8pt_clear.yaml \
  --wandb \
  --wandb-project "MO-PPO-test" \
  --wandb-entity "ttins-tr-ng-i-h-c-c-ng-ngh-i-h-c-qu-c-gia-h-n-i" \
  --total-timesteps 2000000 \
  --num-envs 8

## 7. Đánh giá Mô hình & Xuất Video MP4 có HUD
Chạy 10 episodes kiểm tra tỷ lệ qua màn (Win Rate) và lưu video.

In [ ]:
import os
ckpt_candidates = [
    "/kaggle/working/checkpoints/base_8pt_clear/best_winrate.pt",
    "/kaggle/working/DRL-Mo-Supermario-group-1-test/checkpoints/base_8pt_clear/best_winrate.pt",
    "/kaggle/working/checkpoints/base_8pt_clear/latest.pt",
    "checkpoints/base_8pt_clear/best_winrate.pt",
    "checkpoints/base_8pt_clear/latest.pt"
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), ckpt_candidates[0])
print(f"Sử dụng checkpoint: {ckpt_path}")

!python evaluate.py \
  --checkpoint {ckpt_path} \
  --episodes 10 \
  --save-video \
  --video-dir /kaggle/working/eval_videos

## 8. Xem Video Gameplay Mario trực tiếp trong Kaggle Notebook

In [ ]:
import glob
from IPython.display import HTML, display
from base64 import b64encode

video_files = sorted(
    glob.glob("/kaggle/working/eval_videos/*.mp4") +
    glob.glob("/kaggle/working/checkpoints/base_8pt_clear/eval_videos/*.mp4") +
    glob.glob("checkpoints/base_8pt_clear/eval_videos/*.mp4")
)
if video_files:
    chosen = video_files[-1]
    print(f"🎬 Đang phát video: {chosen}")
    raw_mp4 = open(chosen, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(raw_mp4).decode()
    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("Chưa tìm thấy video nào. Hãy chạy ô đánh giá (bước 7) trước!")

## 9. Đóng gói Checkpoints và Video kết quả tải về máy

In [ ]:
!tar -czvf /kaggle/working/mario_mo_ppo_results.tar.gz \
  /kaggle/working/checkpoints \
  /kaggle/working/eval_videos 2>/dev/null || true
print("✅ Đã tạo file nén mario_mo_ppo_results.tar.gz trong thư mục Output!")